# Pipeline KPI Surete Paris (version simple)

Objectif: construire un score de surete/risque par arrondissement de Paris et, en plus, un score agrege par zone IRIS.

Le pipeline reste simple:

1. chargement des fichiers
2. score delinquance par arrondissement
3. score par zone IRIS
4. agregation finale par arrondissement
5. export CSV

## Comment lire le pipeline

Le notebook suit toujours la meme logique:

1. on charge les fichiers
2. on garde Paris uniquement
3. on calcule un score de risque avec la delinquance
4. on ajoute les cameras et les commissariats
5. on obtient un score final de surete sur 100

Le score final est simple a lire:
- plus il est haut, plus le quartier est considere comme sur
- plus il est bas, plus le quartier est considere comme a risque

La partie la plus importante est la suivante:
- beaucoup de delinquance = risque plus eleve
- beaucoup de cameras = risque plus faible
- commissariat plus proche = risque plus faible

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT_DIR = Path.cwd().resolve().parents[1]
if not (ROOT_DIR / 'data').exists():
    ROOT_DIR = Path.cwd().resolve()
if not (ROOT_DIR / 'data').exists():
    raise FileNotFoundError("Impossible de localiser le dossier data du projet")

def minmax(series):
    smin = series.min()
    smax = series.max()
    if pd.isna(smin) or pd.isna(smax) or smin == smax:
        return pd.Series(0.5, index=series.index)
    return (series - smin) / (smax - smin)

def distance_km(lat1, lon1, lat2, lon2):
    dx = (lon2 - lon1) * 111 * np.cos(np.deg2rad(lat1))
    dy = (lat2 - lat1) * 111
    return np.sqrt(dx**2 + dy**2)

In [2]:
# 1) Chargement des 4 datasets
path_delinquance = ROOT_DIR / 'data' / 'bronze' / 'surete' / 'dataset_delinquances.csv'
path_cameras = ROOT_DIR / 'data' / 'bronze' / 'surete' / 'points.csv'
path_commissariats = ROOT_DIR / 'data' / 'bronze' / 'surete' / 'cartographie-des-emplacements-des-commissariats-a-paris-et-petite-couronne.csv'
path_iris = ROOT_DIR / 'data' / 'bronze' / 'commun' / 'iris.csv'

df_delinq = pd.read_csv(path_delinquance, sep=';', decimal=',', quotechar='"', na_values=['NA'], low_memory=False)
df_cam = pd.read_csv(path_cameras, sep=',', low_memory=False)
df_comm = pd.read_csv(path_commissariats, sep=';', low_memory=False)
df_iris = pd.read_csv(path_iris, sep=';', low_memory=False)

print('Fichiers charges:')
print(f'- Delinquance: {len(df_delinq):,} lignes'.replace(',', ' '))
print(f'- Cameras: {len(df_cam):,} lignes'.replace(',', ' '))
print(f'- Commissariats: {len(df_comm):,} lignes'.replace(',', ' '))
print(f'- IRIS: {len(df_iris):,} lignes'.replace(',', ' '))

Fichiers charges:
- Delinquance: 5 238 000 lignes
- Cameras: 1 339 lignes
- Commissariats: 93 lignes
- IRIS: 992 lignes


In [3]:
# 2) Controle rapide
print('Colonnes delinquance:')
print(df_delinq.columns.tolist())

print('\nColonnes cameras:')
print(df_cam.columns.tolist())

print('\nColonnes commissariats:')
print(df_comm.columns.tolist())

print('\nColonnes IRIS:')
print(df_iris.columns.tolist())

print('\nApercu IRIS:')
display(df_iris.head(3))

Colonnes delinquance:
['CODGEO_2025', 'annee', 'indicateur', 'unite_de_compte', 'nombre', 'taux_pour_mille', 'est_diffuse', 'insee_pop', 'insee_pop_millesime', 'insee_log', 'insee_log_millesime', 'complement_info_nombre', 'complement_info_taux']

Colonnes cameras:
['styleUrl', 'icon', 'LOCALISATI', 'NUMÉRO', 'ARRONDISSE', 'MISE_EN_SE', 'lon', 'lat']

Colonnes commissariats:
['name', 'description', 'geometry', 'geo_point_2d']

Colonnes IRIS:
['Geo Shape', 'DEP', 'INSEE_COM', 'NOM_COM', 'IRIS', 'CODE_IRIS', 'NOM_IRIS', 'TYP_IRIS', 'Geo Point', 'ID']

Apercu IRIS:


,Geo Shape,DEP,INSEE_COM,NOM_COM,IRIS,CODE_IRIS,NOM_IRIS,TYP_IRIS,Geo Point,ID
0,"{""coordinates"": [[[2.348444771796439, 48.86237...",75,75101,Paris 1er Arrondissement,205,751010205,Les Halles 5,A,"48.862297570166575, 2.34534858519305",IRIS____0000000751010205
1,"{""coordinates"": [[[2.339143935838259, 48.86220...",75,75101,Paris 1er Arrondissement,303,751010303,Palais Royal 3,A,"48.862970884171986, 2.335627404567633",IRIS____0000000751010303
2,"{""coordinates"": [[[2.295542880656993, 48.87330...",75,75116,Paris 16e Arrondissement,6411,751166411,Chaillot 11,A,"48.871380349816135, 2.296118003770666",IRIS____0000000751166411


In [3]:
# 3) Score risque delinquance (base)
delinq = df_delinq.copy()

# On garde seulement Paris
delinq['codgeo'] = delinq['CODGEO_2025'].astype(str).str.zfill(5)
delinq = delinq[delinq['codgeo'].str.startswith('751')]
delinq['arrondissement'] = delinq['codgeo'].str[-2:]
delinq = delinq[delinq['arrondissement'].isin([f'{i:02d}' for i in range(1, 21)])]

# Taux a utiliser: taux principal, sinon taux complementaire
delinq['taux_effectif'] = pd.to_numeric(delinq['taux_pour_mille'], errors='coerce')
delinq['taux_effectif'] = delinq['taux_effectif'].fillna(pd.to_numeric(delinq['complement_info_taux'], errors='coerce'))

# Colonnes utiles
delinq['annee'] = pd.to_numeric(delinq['annee'], errors='coerce')
delinq['indicateur'] = delinq['indicateur'].astype(str).str.strip()
delinq = delinq.dropna(subset=['annee', 'taux_effectif'])
delinq['annee'] = delinq['annee'].astype(int)

# On passe en format large: 1 colonne par indicateur
pivot = delinq.pivot_table(
    index=['annee', 'arrondissement'],
    columns='indicateur',
    values='taux_effectif',
    aggfunc='mean'
).sort_index()

# Normalisation simple de chaque indicateur
norm = pivot.apply(minmax, axis=0)
score_delinq = norm.mean(axis=1).mul(100).rename('score_risque_delinq_100').reset_index()
score_delinq['score_surete_delinq_100'] = 100 - score_delinq['score_risque_delinq_100']

print(f'Lignes score delinquance: {len(score_delinq):,}'.replace(',', ' '))
display(score_delinq.head(10))

Lignes score delinquance: 200


,annee,arrondissement,score_risque_delinq_100,score_surete_delinq_100
0,2016,01,47.529767,52.470233
1,2016,02,32.582723,67.417277
2,2016,03,16.932099,83.067901
3,2016,04,19.006984,80.993016
4,2016,05,9.227817,90.772183
5,2016,06,12.621852,87.378148
6,2016,07,9.834178,90.165822
7,2016,08,42.658426,57.341574
8,2016,09,20.984063,79.015937
9,2016,10,22.508819,77.491181


In [4]:
# 3bis) Preparation cameras et commissariats
cam = df_cam.copy()

cam_arr_col = next((c for c in cam.columns if str(c).upper().startswith('ARRONDISSE')), None)
if cam_arr_col is None:
    raise ValueError("Colonne arrondissement non trouvee dans points.csv")

cam['arrondissement'] = (
    cam[cam_arr_col]
    .astype(str)
    .str.extract(r'(\d+)')[0]
    .astype('Int64')
    .pipe(lambda s: s - 75000)
    .astype(str)
    .str.zfill(2)
)
cam = cam[cam['arrondissement'].isin([f'{i:02d}' for i in range(1, 21)])]

arrondissements = (
    cam.groupby('arrondissement', as_index=False)
    .size()
    .rename(columns={'size': 'nb_cameras'})
)
arrondissements['camera_risk_100'] = (1 - minmax(arrondissements['nb_cameras'])) * 100

comm = df_comm.copy()
geo_col = 'geo_point_2d'
if geo_col not in comm.columns:
    raise ValueError("Colonne geo_point_2d non trouvee dans le CSV des commissariats")

geo_comm = comm[geo_col].astype(str).str.split(',', n=1, expand=True)
comm['lat'] = pd.to_numeric(geo_comm[0], errors='coerce')
comm['lon'] = pd.to_numeric(geo_comm[1], errors='coerce')
comm = comm.dropna(subset=['lat', 'lon'])

print('Preparation terminee:')
print(f"- Arrondissements cameras: {len(arrondissements)}")
print(f"- Commissariats geolocalises: {len(comm)}")
display(arrondissements.head())

Preparation terminee:
- Arrondissements cameras: 20
- Commissariats geolocalises: 93


,arrondissement,nb_cameras,camera_risk_100
0,01,54,65.384615
1,02,28,98.717949
2,03,27,100.000000
3,04,56,62.820513
4,05,58,60.256410


In [5]:
# 4) Score par zone geographique (IRIS) puis agregation par arrondissement
iris = df_iris.copy()
iris['codgeo'] = iris['INSEE_COM'].astype(str).str.zfill(5)
iris = iris[iris['codgeo'].str.startswith('751')]
iris['arrondissement'] = iris['codgeo'].str[-2:]
iris = iris[iris['arrondissement'].isin([f'{i:02d}' for i in range(1, 21)])]

# Coordonnees du centre IRIS
geo = iris['Geo Point'].astype(str).str.split(',', n=1, expand=True)
iris['lat'] = pd.to_numeric(geo[0], errors='coerce')
iris['lon'] = pd.to_numeric(geo[1], errors='coerce')
iris = iris.dropna(subset=['lat', 'lon'])

# On rattache le score delinquance de l'arrondissement a chaque IRIS
iris_score = iris[['CODE_IRIS', 'NOM_IRIS', 'arrondissement', 'lat', 'lon']].drop_duplicates().copy()
iris_score = iris_score.merge(score_delinq, on='arrondissement', how='left')

# Cameras: meme score que l'arrondissement, plus simple a lire
iris_score = iris_score.merge(arrondissements[['arrondissement', 'nb_cameras', 'camera_risk_100']], on='arrondissement', how='left')

# Commissariat le plus proche pour chaque IRIS
comm_lat = comm['lat'].to_numpy()
comm_lon = comm['lon'].to_numpy()

def nearest_commissariat(lat, lon):
    if pd.isna(lat) or pd.isna(lon) or len(comm_lat) == 0:
        return np.nan
    dist = distance_km(lat, lon, comm_lat, comm_lon)
    return dist.min()

iris_score['dist_commissariat_km'] = iris_score.apply(lambda r: nearest_commissariat(r['lat'], r['lon']), axis=1)
iris_score['commissariat_risk_100'] = minmax(iris_score['dist_commissariat_km']) * 100

# Score final IRIS
iris_score['score_risque_final_100'] = (0.7 * iris_score['score_risque_delinq_100'] + 0.2 * iris_score['commissariat_risk_100'] + 0.1 * iris_score['camera_risk_100'])
iris_score['score_surete_final_100'] = 100 - iris_score['score_risque_final_100']

# Agregation simple par arrondissement
zone_agregee = iris_score.groupby(['annee', 'arrondissement'], as_index=False).agg(
    score_surete_zone_moyen_100=('score_surete_final_100', 'mean'),
    score_risque_zone_moyen_100=('score_risque_final_100', 'mean'),
    nb_iris=('CODE_IRIS', 'nunique')
)

print(f'Lignes IRIS score: {len(iris_score):,}'.replace(',', ' '))
print(f'Lignes zone aggregee: {len(zone_agregee):,}'.replace(',', ' '))

last_year = int(zone_agregee['annee'].max())
print(f'Derniere annee: {last_year}')
display(zone_agregee[zone_agregee['annee'] == last_year].head(20))

Lignes IRIS score: 9 920
Lignes zone aggregee: 200
Derniere annee: 2025


,annee,arrondissement,score_surete_zone_moyen_100,score_risque_zone_moyen_100,nb_iris
180,2025,01,46.804329,53.195671,17
181,2025,02,65.624306,34.375694,14
182,2025,03,72.928480,27.071520,17
183,2025,04,76.405985,23.594015,21
184,2025,05,80.774861,19.225139,31
185,2025,06,74.782208,25.217792,23
186,2025,07,83.450107,16.549893,36
187,2025,08,66.899838,33.100162,32
188,2025,09,74.136008,25.863992,34
189,2025,10,70.725358,29.274642,42


## 4bis) KPI final unique par quartier avec infos IRIS

> Objectif: une seule sortie surete par quartier, tout en conservant les informations IRIS dans la meme table.

On rattache chaque IRIS au quartier le plus proche (dans le meme arrondissement), puis on calcule:
- les scores moyens quartier
- le nombre d'IRIS rattaches
- des stats IRIS (min/max/ecart-type)
- la liste des codes et noms d'IRIS rattaches

In [6]:
quartiers_path = ROOT_DIR / 'data' / 'bronze' / 'commun' / 'quartier_paris.csv'
quartiers = pd.read_csv(quartiers_path, sep=';', encoding='utf-8-sig')

quartiers = quartiers.rename(
    columns={
        'C_QU': 'code_quartier',
        'C_QUINSEE': 'code_insee_quartier',
        'L_QU': 'nom_quartier',
        'C_AR': 'arrondissement_num',
        'Geometry X Y': 'geometry_xy'
    }
)

quartiers['code_quartier'] = pd.to_numeric(quartiers['code_quartier'], errors='coerce').astype('Int64')
quartiers['code_insee_quartier'] = pd.to_numeric(quartiers['code_insee_quartier'], errors='coerce').astype('Int64')

q_geo = quartiers['geometry_xy'].astype(str).str.split(',', n=1, expand=True)
quartiers['lat_q'] = pd.to_numeric(q_geo[0], errors='coerce')
quartiers['lon_q'] = pd.to_numeric(q_geo[1], errors='coerce')
quartiers['arrondissement'] = (
    pd.to_numeric(quartiers['arrondissement_num'], errors='coerce')
    .astype('Int64')
    .astype(str)
    .str.zfill(2)
)
quartiers = quartiers.dropna(subset=['lat_q', 'lon_q'])

# Mapping de secours code_quartier -> vrai code INSEE quartier (7510xxx).
insee_by_qu = (
    quartiers.dropna(subset=['code_quartier', 'code_insee_quartier'])
    .drop_duplicates(subset=['code_quartier'])
    .set_index('code_quartier')['code_insee_quartier']
    .to_dict()
)

quartiers_keep = quartiers[
    ['code_quartier', 'code_insee_quartier', 'nom_quartier', 'arrondissement', 'lat_q', 'lon_q']
] .drop_duplicates()


def nearest_quartier_in_arr(row):
    subset = quartiers_keep[quartiers_keep['arrondissement'] == row['arrondissement']]
    if subset.empty:
        return pd.Series([pd.NA, pd.NA, pd.NA])
    d = distance_km(row['lat'], row['lon'], subset['lat_q'].to_numpy(), subset['lon_q'].to_numpy())
    idx = int(np.argmin(d))
    ref = subset.iloc[idx]
    return pd.Series([ref['code_quartier'], ref['code_insee_quartier'], ref['nom_quartier']])


def join_sorted_unique(values: pd.Series) -> str:
    unique_vals = sorted({str(v).strip() for v in values.dropna() if str(v).strip()})
    return ' | '.join(unique_vals)


iris_score[['code_quartier_ref', 'code_insee_quartier', 'nom_quartier']] = iris_score.apply(nearest_quartier_in_arr, axis=1)

# Forcer un vrai code INSEE quartier (7510xxx) même si une valeur parasite apparaît.
iris_score['code_quartier_ref'] = pd.to_numeric(iris_score['code_quartier_ref'], errors='coerce').astype('Int64')
iris_score['code_insee_quartier'] = pd.to_numeric(iris_score['code_insee_quartier'], errors='coerce').astype('Int64')
mask_invalid = ~iris_score['code_insee_quartier'].astype('string').str.startswith('75')
iris_score.loc[mask_invalid, 'code_insee_quartier'] = iris_score.loc[mask_invalid, 'code_quartier_ref'].map(insee_by_qu)
iris_score = iris_score.dropna(subset=['code_insee_quartier']).copy()
iris_score['code_insee_quartier'] = iris_score['code_insee_quartier'].astype('Int64').astype(str)

kpi_surete_quartier = (
    iris_score.groupby(['annee', 'arrondissement', 'code_insee_quartier'], as_index=False)
    .agg(
        score_surete_quartier_moyen_100=('score_surete_final_100', 'mean'),
        score_risque_quartier_moyen_100=('score_risque_final_100', 'mean'),
        score_surete_iris_min_100=('score_surete_final_100', 'min'),
        score_surete_iris_max_100=('score_surete_final_100', 'max'),
        score_surete_iris_std_100=('score_surete_final_100', 'std'),
        score_risque_iris_min_100=('score_risque_final_100', 'min'),
        score_risque_iris_max_100=('score_risque_final_100', 'max'),
        score_risque_iris_std_100=('score_risque_final_100', 'std'),
        nb_iris_rattaches=('CODE_IRIS', 'nunique'),
        codes_iris_rattaches=('CODE_IRIS', join_sorted_unique),
        noms_iris_rattaches=('NOM_IRIS', join_sorted_unique),
        dist_commissariat_km_moyenne=('dist_commissariat_km', 'mean'),
        nb_cameras_arrondissement=('nb_cameras', 'max')
    )
)

for c in [
    'score_surete_quartier_moyen_100', 'score_risque_quartier_moyen_100',
    'score_surete_iris_min_100', 'score_surete_iris_max_100', 'score_surete_iris_std_100',
    'score_risque_iris_min_100', 'score_risque_iris_max_100', 'score_risque_iris_std_100',
    'dist_commissariat_km_moyenne'
 ]:
    if c in kpi_surete_quartier.columns:
        kpi_surete_quartier[c] = kpi_surete_quartier[c].round(4)

print(f'Lignes KPI surete quartier (enrichi IRIS): {len(kpi_surete_quartier):,}'.replace(',', ' '))
display(kpi_surete_quartier.head(12))

Lignes KPI surete quartier (enrichi IRIS): 800


,annee,arrondissement,code_insee_quartier,score_surete_quartier_moyen_100,score_risque_quartier_moyen_100,score_surete_iris_min_100,score_surete_iris_max_100,score_surete_iris_std_100,score_risque_iris_min_100,score_risque_iris_max_100,score_risque_iris_std_100,nb_iris_rattaches,codes_iris_rattaches,noms_iris_rattaches,dist_commissariat_km_moyenne,nb_cameras_arrondissement
0,2016,01,7510101,52.9432,47.0568,52.1586,53.7278,1.1096,46.2722,47.8414,1.1096,2,751010104 | 751010199,Saint-Germain l'Auxerrois 4 | Seine et Berges,0.8105,54
1,2016,01,7510102,57.0605,42.9395,54.1663,59.7779,1.6413,40.2221,45.8337,1.6413,9,751010101 | 751010102 | 751010103 | 751010201 ...,Les Halles 1 | Les Halles 2 | Les Halles 3 | L...,0.3658,54
2,2016,01,7510103,55.7880,44.2120,55.2840,56.6722,0.7683,43.3278,44.7160,0.7683,3,751010301 | 751010302 | 751010303,Palais Royal 1 | Palais Royal 2 | Palais Royal 3,0.5032,54
3,2016,01,7510104,57.4220,42.5780,55.7461,58.8650,1.5724,41.1350,44.2539,1.5724,3,751010105 | 751010401 | 751010402,Place Vendôme 1 | Place Vendôme 2 | Tuileries,0.3267,54
4,2016,02,7510201,64.5300,35.4700,63.3667,65.4403,1.0597,34.5597,36.6333,1.0597,3,751020501 | 751020502 | 751020503,Gaillon 1 | Gaillon 2 | Gaillon 3,0.3291,28
5,2016,02,7510202,64.2334,35.7666,63.9664,64.5004,0.3776,35.4996,36.0336,0.3776,2,751020601 | 751020602,Vivienne 1 | Vivienne 2,0.3611,28
6,2016,02,7510203,65.6023,34.3977,64.1974,66.6042,1.0183,33.3958,35.8026,1.0183,4,751020701 | 751020702 | 751020703 | 751020704,Mail 1 | Mail 2 | Mail 3 | Mail 4,0.2133,28
7,2016,02,7510204,64.0147,35.9853,63.2454,64.4974,0.4723,35.5026,36.7546,0.4723,5,751020801 | 751020802 | 751020803 | 751020804 ...,Bonne Nouvelle 1 | Bonne Nouvelle 2 | Bonne No...,0.3847,28
8,2016,03,7510301,73.0310,26.9690,71.4662,74.5839,1.4563,25.4161,28.5338,1.4563,5,751030901 | 751030902 | 751030903 | 751030904 ...,Arts et Métiers 1 | Arts et Métiers 2 | Arts e...,0.5803,27
9,2016,03,7510302,70.4635,29.5365,68.9061,71.9418,1.3012,28.0582,31.0939,1.3012,4,751031001 | 751031002 | 751031003 | 751031004,Enfants Rouges 1 | Enfants Rouges 2 | Enfants ...,0.8576,27


In [7]:
# 5) Export harmonise: une seule sortie surete par quartier (avec infos IRIS)
output_dir = ROOT_DIR / 'data' / 'outputs'
output_dir.mkdir(exist_ok=True)

output_path_quartier = output_dir / 'kpi_score_surete_quartier_estime_depuis_iris.csv'
kpi_surete_quartier.to_csv(output_path_quartier, index=False)

print(f'Fichier cree: {output_path_quartier}')
print(f'Nombre de lignes quartier: {len(kpi_surete_quartier):,}'.replace(',', ' '))
print('Sorties arrondissement et iris detail desactivees pour harmonisation.')

Fichier cree: C:\Users\rayan\Desktop\efrei\projet-data-dev\data\outputs\kpi_score_surete_quartier_estime_depuis_iris.csv
Nombre de lignes quartier: 800
Sorties arrondissement et iris detail desactivees pour harmonisation.
